In [5]:
from src.database.connection import init_db
from src.ingestion.fundamental_loader import load_financials_or_raise, load_dividends_or_raise
from src.database.db_manager import upsert_financials, upsert_dividends, db_summary

init_db()   # creates the two new tables automatically

fin_df = load_financials_or_raise()
upsert_financials(fin_df)

div_df = load_dividends_or_raise()
upsert_dividends(div_df)

print(db_summary())

  [financials_loader] Reading 'sample_financials_2025_raw.csv' ...
  [financials_loader] Raw columns: ['ticker', 'fiscal_year', 'period_type', 'revenue (Billions)', 'net_profit (Billions)', 'total_assets (Billions)', 'total_equity (Billions)', 'total_debt (Billions)', 'eps (PKR)', 'book_value_per_share (PKR)', 'shares_outstanding (Millions)', 'source']
  [financials_loader] Loaded 40 rows for 40 ticker(s) from 'sample_financials_2025_raw.csv'
Load ✅ PASSED  (40 in → 40 out)
  Warnings:
    • 1 rows have zero revenue (kept — may be valid)
    • 2 rows have negative total_equity (kept — technically valid)
    • Clean data: 40 rows | 40 ticker(s) | fiscal years: [2025]
  [upsert_financials] Inserting 40 rows in 1 batch(es) ...
  [upsert_financials] batch 1/1 (40/40 rows)
  [upsert_financials] Done — 40 rows upserted.
  [dividends_loader] Reading 'sample_dividends_2025.csv' ...
  [dividends_loader] Raw columns: ['ticker', 'ex_dividend_date', 'fiscal_year', 'dividend_per_share', 'dividend_t

In [7]:
from src.database.db_manager import load_financials, load_dividends

fin = load_financials()
div = load_dividends()

print("Financial rows:", len(fin))
print("Dividend rows:", len(div))

display(fin.head())
display(div.head())

Financial rows: 40
Dividend rows: 40


,ticker,fiscal_year,period_type,revenue,net_profit,total_assets,total_equity,total_debt,eps,book_value_per_share,shares_outstanding,source
0,ABL,2025,ANNUAL,1.556000e+11,4.210000e+10,2.120000e+12,1.701000e+11,1.949900e+12,36.80,148.6,1.145000e+09,FY25 Audited/Announced
1,AIRLINK,2025,ANNUAL,9.540000e+10,4.200000e+09,4.500000e+10,1.700000e+10,2.800000e+10,10.60,43.0,3.950000e+08,FY25 Audited/Announced
2,BAHL,2025,ANNUAL,2.105000e+11,4.120000e+10,2.550000e+12,1.654000e+11,2.384600e+12,37.10,148.8,1.111000e+09,FY25 Audited/Announced
3,BOP,2025,ANNUAL,9.820000e+10,1.150000e+10,1.620000e+12,7.850000e+10,1.541500e+12,3.52,24.1,3.267000e+09,FY25 Audited/Announced
4,CHCC,2025,ANNUAL,4.250000e+10,6.800000e+09,8.500000e+10,4.000000e+10,4.500000e+10,35.10,206.0,1.940000e+08,FY25 Audited/Announced


,ticker,ex_dividend_date,fiscal_year,dividend_per_share,dividend_type,source
0,ABL,2025-12-31,2025,9.0,Cash,SAMPLE_ESTIMATE
1,AIRLINK,2025-12-31,2025,5.0,Cash,SAMPLE_ESTIMATE
2,BAHL,2025-12-31,2025,10.0,Cash,SAMPLE_ESTIMATE
3,BOP,2025-12-31,2025,2.5,Cash,SAMPLE_ESTIMATE
4,CHCC,2025-12-31,2025,7.5,Cash,SAMPLE_ESTIMATE


In [8]:
fin[fin["ticker"] == "HBL"].T

,13
ticker,HBL
fiscal_year,2025
period_type,ANNUAL
revenue,361200000000.0
net_profit,66800000000.0
total_assets,7720000000000.0
total_equity,681500000000.0
total_debt,7038500000000.0
eps,45.48
book_value_per_share,464.3


In [6]:
from src.analysis.fundamental_ratios import (
    compute_all_ratios,
    compute_ratios_for_ticker,
    flag_ratio_quality,
)

# ── All tickers ───────────────────────────────────────────────────
ratios_df = compute_all_ratios()
display(ratios_df)

# ── With quality flags ────────────────────────────────────────────
flagged_df = flag_ratio_quality(ratios_df)
flag_cols  = ["ticker", "pe_ratio", "pe_flag",
              "roe", "roe_flag", "debt_to_equity", "debt_flag",
              "profit_margin", "margin_flag", "revenue_growth_yoy", "growth_flag"]
flag_cols  = [c for c in flag_cols if c in flagged_df.columns]
display(flagged_df[flag_cols])

# ── Single ticker ─────────────────────────────────────────────────
single = compute_ratios_for_ticker("HBL")
for k, v in single.items():
    print(f"  {k:<25} {v}")

[ratio_engine] Computed ratios for 25 ticker(s) | fiscal year range: 2025–2025


,ticker,latest_close,fiscal_year,pe_ratio,pb_ratio,roe,profit_margin,debt_to_equity,dividend_yield,revenue_growth_yoy,eps_growth_yoy
0,ABL,165.04,2025,4.48,1.11,24.75,27.06,11.46,5.45,NaN,NaN
1,CHCC,332.53,2025,9.47,1.61,17.00,16.00,1.12,2.26,NaN,NaN
2,COLG,1286.92,2025,21.96,7.80,35.50,12.35,0.62,13.99,NaN,NaN
3,DGKC,234.28,2025,24.46,2.06,8.40,6.42,1.90,2.56,NaN,NaN
4,EFERT,214.62,2025,12.66,4.10,32.29,9.53,1.65,9.32,NaN,NaN
5,ENGRO,485.38,2025,7.93,1.04,13.04,7.25,2.16,3.30,NaN,NaN
6,FATIMA,137.99,2025,15.66,2.42,15.42,9.97,0.80,5.80,NaN,NaN
7,FFC,477.28,2025,9.23,5.01,54.52,17.02,2.17,5.87,NaN,NaN
8,HBL,166.02,2025,3.65,0.36,9.80,18.49,10.33,8.43,NaN,NaN
9,INDU,2076.06,2025,6.43,2.17,33.87,11.81,0.60,3.85,NaN,NaN


,ticker,pe_ratio,pe_flag,roe,roe_flag,debt_to_equity,debt_flag,profit_margin,margin_flag,revenue_growth_yoy,growth_flag
0,ABL,4.48,Cheap,24.75,Strong,11.46,High,27.06,Healthy,NaN,N/A
1,CHCC,9.47,Cheap,17.00,Strong,1.12,Moderate,16.00,Healthy,NaN,N/A
2,COLG,21.96,Expensive,35.50,Strong,0.62,Moderate,12.35,Thin,NaN,N/A
3,DGKC,24.46,Expensive,8.40,Moderate,1.90,High,6.42,Thin,NaN,N/A
4,EFERT,12.66,Fair,32.29,Strong,1.65,High,9.53,Thin,NaN,N/A
5,ENGRO,7.93,Cheap,13.04,Moderate,2.16,High,7.25,Thin,NaN,N/A
6,FATIMA,15.66,Fair,15.42,Strong,0.80,Moderate,9.97,Thin,NaN,N/A
7,FFC,9.23,Cheap,54.52,Strong,2.17,High,17.02,Healthy,NaN,N/A
8,HBL,3.65,Cheap,9.80,Moderate,10.33,High,18.49,Healthy,NaN,N/A
9,INDU,6.43,Cheap,33.87,Strong,0.60,Moderate,11.81,Thin,NaN,N/A


[ratio_engine] Computed ratios for 25 ticker(s) | fiscal year range: 2025–2025
  ticker                    HBL
  latest_close              166.02
  fiscal_year               2025
  pe_ratio                  3.65
  pb_ratio                  0.36
  roe                       9.8
  profit_margin             18.49
  debt_to_equity            10.33
  dividend_yield            8.43
  revenue_growth_yoy        nan
  eps_growth_yoy            nan


In [10]:
from src.analysis.fundamentals import build_fundamental_metrics

fundamental_df = build_fundamental_metrics()

print(fundamental_df.shape)
fundamental_df.head()

[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
(53, 21)


,ticker,fiscal_year,latest_close,revenue,net_profit,total_assets,total_equity,total_debt,eps,book_value_per_share,...,annual_dividend_per_share,pe_ratio,pb_ratio,roe,debt_to_equity,dividend_yield,net_profit_margin,payout_ratio,data_quality_score,fundamental_notes
0,ABL,2025.0,165.04,1.556000e+11,4.210000e+10,2.120000e+12,1.701000e+11,1.949900e+12,36.8,148.6,...,9.0,4.4848,1.1106,24.7501,11.4633,5.4532,27.0566,24.4565,100,Complete data
1,AIRLINK,2025.0,NaN,9.540000e+10,4.200000e+09,4.500000e+10,1.700000e+10,2.800000e+10,10.6,43.0,...,5.0,NaN,NaN,24.7059,1.6471,NaN,4.4025,47.1698,85,Missing latest price
2,ATLH,NaN,1398.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
3,BAFL,NaN,44.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
4,BAHL,2025.0,NaN,2.105000e+11,4.120000e+10,2.550000e+12,1.654000e+11,2.384600e+12,37.1,148.8,...,10.0,NaN,NaN,24.9093,14.4172,NaN,19.5724,26.9542,85,Missing latest price


In [11]:
fundamental_df[
    [
        "ticker",
        "latest_close",
        "eps",
        "book_value_per_share",
        "annual_dividend_per_share",
        "pe_ratio",
        "pb_ratio",
        "roe",
        "debt_to_equity",
        "dividend_yield",
        "net_profit_margin",
        "payout_ratio",
        "data_quality_score",
        "fundamental_notes",
    ]
].head(20)

,ticker,latest_close,eps,book_value_per_share,annual_dividend_per_share,pe_ratio,pb_ratio,roe,debt_to_equity,dividend_yield,net_profit_margin,payout_ratio,data_quality_score,fundamental_notes
0,ABL,165.04,36.80,148.6,9.0,4.4848,1.1106,24.7501,11.4633,5.4532,27.0566,24.4565,100,Complete data
1,AIRLINK,NaN,10.60,43.0,5.0,NaN,NaN,24.7059,1.6471,NaN,4.4025,47.1698,85,Missing latest price
2,ATLH,1398.07,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
3,BAFL,44.44,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
4,BAHL,NaN,37.10,148.8,10.0,NaN,NaN,24.9093,14.4172,NaN,19.5724,26.9542,85,Missing latest price
5,BOP,NaN,3.52,24.1,2.5,NaN,NaN,14.6497,19.6369,NaN,11.7108,71.0227,85,Missing latest price
6,CHCC,332.53,35.10,206.0,7.5,9.4738,1.6142,17.0000,1.1250,2.2554,16.0000,21.3675,100,Complete data
7,COLG,1286.92,58.60,165.0,180.0,21.9611,7.7995,35.5000,0.6250,13.9869,12.3478,307.1672,100,Complete data
8,DGKC,234.28,9.58,114.0,6.0,24.4551,2.0551,8.4000,1.9000,2.5610,6.4220,62.6305,100,Complete data
9,EFERT,214.62,16.95,52.4,20.0,12.6619,4.0958,32.2857,1.6486,9.3188,9.5318,117.9941,100,Complete data


In [12]:
fundamental_df[fundamental_df["ticker"] == "HBL"].T

,19
ticker,HBL
fiscal_year,2025.0
latest_close,166.02
revenue,361200000000.0
net_profit,66800000000.0
total_assets,7720000000000.0
total_equity,681500000000.0
total_debt,7038500000000.0
eps,45.48
book_value_per_share,464.3


In [14]:
from src.analysis.fundamentals import (
    build_fundamental_metrics,
    get_fundamental_for_ticker,
)

# ── Full table ────────────────────────────────────────────────────
metrics_df = build_fundamental_metrics()
display(metrics_df)

# ── Inspect quality scores ────────────────────────────────────────
display(
    metrics_df[["ticker", "data_quality_score", "fundamental_notes"]]
    .sort_values("data_quality_score")
)

# ── Check a known negative company ───────────────────────────────
for tk in ["SSGC", "KEL", "PSMC"]:
    result = get_fundamental_for_ticker(tk)
    print(f"\n{tk}")
    for k, v in result.items():
        print(f"  {k:<30} {v}")

# ── Ratio sanity check ────────────────────────────────────────────
ratio_cols = [
    "ticker", "pe_ratio", "pb_ratio", "roe",
    "debt_to_equity", "dividend_yield",
    "net_profit_margin", "payout_ratio",
]
display(metrics_df[ratio_cols])

[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100


,ticker,fiscal_year,latest_close,revenue,net_profit,total_assets,total_equity,total_debt,eps,book_value_per_share,...,annual_dividend_per_share,pe_ratio,pb_ratio,roe,debt_to_equity,dividend_yield,net_profit_margin,payout_ratio,data_quality_score,fundamental_notes
0,ABL,2025.0,165.04,1.556000e+11,4.210000e+10,2.120000e+12,1.701000e+11,1.949900e+12,36.80,148.60,...,9.0,4.4848,1.1106,24.7501,11.4633,5.4532,27.0566,24.4565,100,Complete data
1,AIRLINK,2025.0,NaN,9.540000e+10,4.200000e+09,4.500000e+10,1.700000e+10,2.800000e+10,10.60,43.00,...,5.0,NaN,NaN,24.7059,1.6471,NaN,4.4025,47.1698,85,Missing latest price
2,ATLH,NaN,1398.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
3,BAFL,NaN,44.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
4,BAHL,2025.0,NaN,2.105000e+11,4.120000e+10,2.550000e+12,1.654000e+11,2.384600e+12,37.10,148.80,...,10.0,NaN,NaN,24.9093,14.4172,NaN,19.5724,26.9542,85,Missing latest price
5,BOP,2025.0,NaN,9.820000e+10,1.150000e+10,1.620000e+12,7.850000e+10,1.541500e+12,3.52,24.10,...,2.5,NaN,NaN,14.6497,19.6369,NaN,11.7108,71.0227,85,Missing latest price
6,CHCC,2025.0,332.53,4.250000e+10,6.800000e+09,8.500000e+10,4.000000e+10,4.500000e+10,35.10,206.00,...,7.5,9.4738,1.6142,17.0000,1.1250,2.2554,16.0000,21.3675,100,Complete data
7,COLG,2025.0,1286.92,1.150000e+11,1.420000e+10,6.500000e+10,4.000000e+10,2.500000e+10,58.60,165.00,...,180.0,21.9611,7.7995,35.5000,0.6250,13.9869,12.3478,307.1672,100,Complete data
8,DGKC,2025.0,234.28,6.540000e+10,4.200000e+09,1.450000e+11,5.000000e+10,9.500000e+10,9.58,114.00,...,6.0,24.4551,2.0551,8.4000,1.9000,2.5610,6.4220,62.6305,100,Complete data
9,EFERT,2025.0,214.62,2.371000e+11,2.260000e+10,1.854000e+11,7.000000e+10,1.154000e+11,16.95,52.40,...,20.0,12.6619,4.0958,32.2857,1.6486,9.3188,9.5318,117.9941,100,Complete data


,ticker,data_quality_score,fundamental_notes
18,GLAXO,45,Missing financial data | Missing dividend data
33,NCL,45,Missing financial data | Missing dividend data
25,KTML,45,Missing financial data | Missing dividend data
22,ICI,45,Missing financial data | Missing dividend data
17,GATM,45,Missing financial data | Missing dividend data
15,FFBL,45,Missing financial data | Missing dividend data
14,FEROZ,45,Missing financial data | Missing dividend data
45,SEARL,45,Missing financial data | Missing dividend data
46,SITC,45,Missing financial data | Missing dividend data
36,NML,45,Missing financial data | Missing dividend data


[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100

SSGC
  ticker                         SSGC
  fiscal_year                    2025.0
  latest_close                   61.49
  revenue                        520000000000.0
  net_profit                     -8500000000.0
  total_assets                   580000000000.0
  total_equity                   -30000000000.0
  total_debt                     610000000000.0
  eps                            -9.6
  book_value_per_share           -34.0
  shares_outstanding             881000000.0
  annual_dividend_per_share      0.0
  pe_ratio                       None
  pb_ratio                       None
  roe                            None
  debt_to_equity                 None
  dividend_yield                 0.0
  net_profit_margin              -1.6346
  payout_ratio                   None
  data_quality_score             80
  fundamental_notes              Negative EPS | Negative equity
[fundamental] Built metrics for 53 t

,ticker,pe_ratio,pb_ratio,roe,debt_to_equity,dividend_yield,net_profit_margin,payout_ratio
0,ABL,4.4848,1.1106,24.7501,11.4633,5.4532,27.0566,24.4565
1,AIRLINK,NaN,NaN,24.7059,1.6471,NaN,4.4025,47.1698
2,ATLH,NaN,NaN,NaN,NaN,0.0000,NaN,NaN
3,BAFL,NaN,NaN,NaN,NaN,0.0000,NaN,NaN
4,BAHL,NaN,NaN,24.9093,14.4172,NaN,19.5724,26.9542
5,BOP,NaN,NaN,14.6497,19.6369,NaN,11.7108,71.0227
6,CHCC,9.4738,1.6142,17.0000,1.1250,2.2554,16.0000,21.3675
7,COLG,21.9611,7.7995,35.5000,0.6250,13.9869,12.3478,307.1672
8,DGKC,24.4551,2.0551,8.4000,1.9000,2.5610,6.4220,62.6305
9,EFERT,12.6619,4.0958,32.2857,1.6486,9.3188,9.5318,117.9941


In [1]:
from src.analysis.verdict import get_verdict, apply_verdicts

# Should print: Strong Buy 🟢
v = get_verdict(85)
print(v.label, v.emoji)

# Should print: Avoid 🔴
v = get_verdict(15)
print(v.label, v.emoji)

Strong Buy 🟢
Avoid 🔴


In [2]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run(force_reseed=False)
df = engine.get_screener_df()
print(df[["ticker", "pe_ratio", "roe", "verdict"]].head(10))

[engine] Initialising database...
[engine] DB already has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing indicators and scores...
[engine] Merging fundamental metrics...
[engine] Warning: fundamental merge failed (No module named 'src.analysis.fundamental'). Continuing with technical-only table.
[engine] Ready — 38 tickers | 80,701 price rows | 40 tickers with fundamentals
   ticker pe_ratio   roe     verdict
0     PSO     None  None  Strong Buy
1   ENGRO     None  None  Strong Buy
2    OGDC     None  None         Buy
3     PPL     None  None         Buy
4     FFC     None  None         Buy
5    FFBL     None  None         Buy
6    TELE     None  None         Buy
7    ATLH     None  None         Buy
8     ABL     None  None         Buy
9  FATIMA     None  None        Hold


In [3]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run(force_reseed=False)

df = engine.get_screener_df()

# Confirm fundamental columns are present
fund_cols = [
    "pe_ratio", "pb_ratio", "roe", "debt_to_equity",
    "dividend_yield", "net_profit_margin", "payout_ratio",
    "data_quality_score", "fundamental_notes",
]
print(df[["ticker"] + fund_cols].head(15))

# Confirm technical columns still intact
tech_cols = [
    "ticker", "name", "sector", "latest_close",
    "sma_20", "sma_50", "rsi_14", "demo_score", "verdict",
]
print(df[tech_cols].head(15))

[engine] Initialising database...
[engine] DB already has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing indicators and scores...
[engine] Merging fundamental metrics...
[engine] Warning: fundamental merge failed (No module named 'src.analysis.fundamental'). Continuing with technical-only table.
[engine] Ready — 38 tickers | 80,701 price rows | 40 tickers with fundamentals
    ticker pe_ratio pb_ratio   roe debt_to_equity dividend_yield  \
0      PSO     None     None  None           None           None   
1    ENGRO     None     None  None           None           None   
2     OGDC     None     None  None           None           None   
3      PPL     None     None  None           None           None   
4      FFC     None     None  None           None           None   
5     FFBL     None     None  None           None           None   
6     TELE     None     None  None           None           None   
7     ATLH     None     None  None      

In [1]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run(force_reseed=False)

df = engine.get_screener_df()

# ── Confirm all four score columns ────────────────────────────────
score_cols = [
    "ticker",
    "technical_score",
    "fundamental_score",
    "risk_score",
    "composite_score",
    "demo_score",          # backward-compat alias
    "final_verdict",
    "verdict",             # backward-compat alias
]
print(df[score_cols].head(15).to_string())

# ── Check negative companies don't crash ─────────────────────────
for tk in ["SSGC", "KEL", "PSMC"]:
    row = df[df["ticker"] == tk]
    if not row.empty:
        r = row.iloc[0]
        print(
            f"{tk:6s}  tech={r.technical_score:.1f}  "
            f"fund={r.fundamental_score:.1f}  "
            f"risk={r.risk_score:.1f}  "
            f"comp={r.composite_score:.1f}  "
            f"{r.final_verdict}"
        )

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[engine] Warning: fundamental merge failed (No module named 'src.analysis.fundamental'). Continuing with technical-only table.
[engine] Running three-pillar scoring...
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals
    ticker  technical_score  fundamental_score  risk_score  composite_score  demo_score final_verdict verdict
0      PSO            81.04               50.0       34.96            65.42       65.42           Buy     Buy
1     ATLH            75.05               50.0       32.74            63.47       63.47          Hold    Hold
2      FFC            74.44               50.0       33.52            63.07       63.07          Hold    Hold
3      PPL            73.53               50.0       36.68            62.08       62.08   

In [1]:
from src.analysis.fundamentals import build_fundamental_metrics

fund_df = build_fundamental_metrics()

display(fund_df.head())
print(fund_df.columns.tolist())
print(fund_df[["pe_ratio", "pb_ratio", "roe", "dividend_yield"]].notna().sum())

[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100


,ticker,fiscal_year,latest_close,revenue,net_profit,total_assets,total_equity,total_debt,eps,book_value_per_share,...,annual_dividend_per_share,pe_ratio,pb_ratio,roe,debt_to_equity,dividend_yield,net_profit_margin,payout_ratio,data_quality_score,fundamental_notes
0,ABL,2025.0,165.04,1.556000e+11,4.210000e+10,2.120000e+12,1.701000e+11,1.949900e+12,36.8,148.6,...,9.0,4.4848,1.1106,24.7501,11.4633,5.4532,27.0566,24.4565,100,Complete data
1,AIRLINK,2025.0,NaN,9.540000e+10,4.200000e+09,4.500000e+10,1.700000e+10,2.800000e+10,10.6,43.0,...,5.0,NaN,NaN,24.7059,1.6471,NaN,4.4025,47.1698,85,Missing latest price
2,ATLH,NaN,1398.07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
3,BAFL,NaN,44.44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,45,Missing financial data | Missing dividend data
4,BAHL,2025.0,NaN,2.105000e+11,4.120000e+10,2.550000e+12,1.654000e+11,2.384600e+12,37.1,148.8,...,10.0,NaN,NaN,24.9093,14.4172,NaN,19.5724,26.9542,85,Missing latest price


['ticker', 'fiscal_year', 'latest_close', 'revenue', 'net_profit', 'total_assets', 'total_equity', 'total_debt', 'eps', 'book_value_per_share', 'shares_outstanding', 'annual_dividend_per_share', 'pe_ratio', 'pb_ratio', 'roe', 'debt_to_equity', 'dividend_yield', 'net_profit_margin', 'payout_ratio', 'data_quality_score', 'fundamental_notes']
pe_ratio          23
pb_ratio          23
roe               38
dividend_yield    38
dtype: int64


In [2]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run()

df = engine.get_screener_df()

display(df[[
    "ticker",
    "sector",
    "composite_score",
    "sector_avg_composite_score",
    "composite_score_vs_sector",
    "risk_score",
    "sector_avg_risk_score",
    "risk_score_vs_sector",
    "sector_value_label",
]].head(10))

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals


,ticker,sector,composite_score,sector_avg_composite_score,composite_score_vs_sector,risk_score,sector_avg_risk_score,risk_score_vs_sector,sector_value_label
0,PPL,Energy,73.76,59.976000,0.229825,9.77,24.254000,-0.597180,Attractive vs Sector
1,FFC,Fertilizer,72.65,64.365000,0.128719,18.97,24.445000,-0.223972,Sector Value Leader
2,OGDC,Energy,71.06,59.976000,0.184807,7.84,24.254000,-0.676754,Attractive vs Sector
3,ABL,Banking,70.81,56.528000,0.252654,24.34,28.358000,-0.141688,Sector Value Leader
4,PSO,Energy,65.39,59.976000,0.090269,25.57,24.254000,0.054259,Attractive vs Sector
5,FATIMA,Fertilizer,63.80,64.365000,-0.008778,16.19,24.445000,-0.337697,In Line with Sector
6,ATLH,Automobile,63.34,55.233333,0.146771,32.74,29.926667,0.094008,Attractive vs Sector
7,FFBL,Fertilizer,61.52,64.365000,-0.044201,44.00,24.445000,0.799959,Weak vs Sector
8,MLCF,Cement,61.09,56.710000,0.077235,12.64,15.520000,-0.185567,Sector Value Leader
9,UBL,Banking,61.06,56.528000,0.080173,27.95,28.358000,-0.014387,Attractive vs Sector


In [3]:
from src.analysis.sector_benchmark import build_sector_benchmarks

sector_summary, stock_sector = build_sector_benchmarks()

display(sector_summary)
display(stock_sector[[
    "ticker",
    "sector",
    "sector_avg_composite_score",
    "composite_score_vs_sector",
    "sector_avg_risk_score",
    "risk_score_vs_sector",
    "sector_value_label",
]].head(10))

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals


,sector,sector_stock_count,sector_valid_pe_count,sector_avg_pe,sector_std_pe,sector_median_pe,sector_valid_pb_count,sector_avg_pb,sector_std_pb,sector_valid_roe_count,...,sector_std_roe,sector_valid_dividend_yield_count,sector_avg_dividend_yield,sector_std_dividend_yield,sector_valid_composite_score_count,sector_avg_composite_score,sector_std_composite_score,sector_valid_risk_score_count,sector_avg_risk_score,sector_std_risk_score
0,Automobile,3,1,NaN,NaN,NaN,1,NaN,NaN,1,...,NaN,3,1.284500,2.224819,3,55.233333,11.171599,3,29.926667,21.120997
1,Banking,5,4,4.992400,1.910997,4.25320,4,0.859525,0.375148,4,...,6.374310,5,6.778300,4.904983,5,56.528000,9.640276,5,28.358000,4.402189
2,Cement,5,5,10.921860,8.259857,9.47380,5,1.469160,0.616989,5,...,5.218087,5,3.795280,2.842216,5,56.710000,3.689776,5,15.520000,3.602270
3,Chemicals,3,1,NaN,NaN,NaN,1,NaN,NaN,1,...,NaN,3,1.098800,1.903177,3,51.953333,10.694678,3,33.573333,4.240546
4,Energy,5,4,6.836425,3.870304,7.38115,4,0.913100,0.601744,4,...,4.681887,5,4.027080,3.053224,5,59.976000,15.093238,5,24.254000,18.543269
5,FMCG,2,2,17.471400,6.349395,17.47140,2,8.499300,0.989667,2,...,25.102291,2,9.150350,6.839915,2,55.690000,4.681047,2,12.370000,3.606245
6,Fertilizer,4,3,12.519433,3.217067,12.66190,3,3.840200,1.314571,3,...,19.612119,4,5.245725,3.864260,4,64.365000,5.797129,4,24.445000,13.095156
7,Pharma,3,0,NaN,NaN,NaN,0,NaN,NaN,0,...,NaN,3,0.000000,0.000000,3,56.066667,3.926593,3,38.226667,1.430012
8,Technology,4,3,13.072500,8.102739,9.93460,3,2.185367,2.467570,3,...,8.318954,4,1.562850,2.115015,4,54.982500,1.208453,4,21.375000,15.221477
9,Textiles,4,0,NaN,NaN,NaN,0,NaN,NaN,0,...,NaN,4,0.000000,0.000000,4,47.817500,7.597071,4,38.822500,3.572603


,ticker,sector,sector_avg_composite_score,composite_score_vs_sector,sector_avg_risk_score,risk_score_vs_sector,sector_value_label
0,PPL,Energy,59.976000,0.229825,24.254000,-0.597180,Attractive vs Sector
1,FFC,Fertilizer,64.365000,0.128719,24.445000,-0.223972,Sector Value Leader
2,OGDC,Energy,59.976000,0.184807,24.254000,-0.676754,Attractive vs Sector
3,ABL,Banking,56.528000,0.252654,28.358000,-0.141688,Sector Value Leader
4,PSO,Energy,59.976000,0.090269,24.254000,0.054259,Attractive vs Sector
5,FATIMA,Fertilizer,64.365000,-0.008778,24.445000,-0.337697,In Line with Sector
6,ATLH,Automobile,55.233333,0.146771,29.926667,0.094008,Attractive vs Sector
7,FFBL,Fertilizer,64.365000,-0.044201,24.445000,0.799959,Weak vs Sector
8,MLCF,Cement,56.710000,0.077235,15.520000,-0.185567,Sector Value Leader
9,UBL,Banking,56.528000,0.080173,28.358000,-0.014387,Attractive vs Sector


In [4]:
required_cols = [
    "sector_stock_count",
    "sector_avg_pe",
    "sector_median_pe",
    "sector_avg_pb",
    "sector_avg_roe",
    "sector_avg_dividend_yield",
    "sector_avg_composite_score",
    "sector_avg_risk_score",
    "pe_vs_sector",
    "pb_vs_sector",
    "roe_vs_sector",
    "dividend_yield_vs_sector",
    "composite_score_vs_sector",
    "risk_score_vs_sector",
    "pe_sector_z",
    "pb_sector_z",
    "roe_sector_z",
    "dividend_yield_sector_z",
    "composite_score_sector_z",
    "risk_score_sector_z",
    "sector_value_label",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing columns:", missing)

Missing columns: []


In [1]:
from src.screener.presets import get_preset_names, apply_preset

preset_names = get_preset_names()
preset_names

['Strong Buy Candidates',
 'Undervalued Quality',
 'Dividend Picks',
 'Momentum Leaders',
 'Low Risk Blue Chips',
 'Oversold Bounce',
 'Avoid / High Risk',
 'Sector Relative Value']

In [2]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run()

df = engine.get_screener_df()
df.head()

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,5,5,4,4,5,4,1,1,2.213000e+11,7.058000e+11
1,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11
2,OGDC,Oil & Gas Dev. Company,Energy,417.47,402.31,368.23,304.85,67.58,-0.7895,0.0937,...,5,5,4,4,5,4,1,1,2.450000e+11,1.005000e+12
3,ABL,Allied Bank Ltd,Banking,165.04,159.21,154.50,161.53,64.50,0.5718,0.1045,...,5,5,4,4,5,4,1,0,1.949900e+12,1.701000e+11
4,PSO,Pakistan State Oil,Energy,412.67,395.61,384.48,373.85,62.11,2.4393,0.0981,...,5,5,4,4,5,4,1,1,7.354000e+11,2.500000e+11


In [3]:
engine.get_preset_names()

['Strong Buy Candidates',
 'Undervalued Quality',
 'Dividend Picks',
 'Momentum Leaders',
 'Low Risk Blue Chips',
 'Oversold Bounce',
 'Avoid / High Risk',
 'Sector Relative Value']

In [4]:
for preset in engine.get_preset_names():
    out = engine.get_preset(preset)
    print(f"{preset}: {len(out)} rows")
    if not out.empty:
        display(out.head())

Strong Buy Candidates: 0 rows
Undervalued Quality: 0 rows
Dividend Picks: 11 rows


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,MCB,MCB Bank Ltd,Banking,221.59,233.42,223.57,264.85,42.51,-2.6521,-0.0293,...,5,5,4,4,5,4,1,0,3.117200e+12,3.328000e+11
1,LUCK,Lucky Cement Ltd,Cement,452.32,459.93,456.17,748.57,46.89,-1.4157,-0.0265,...,5,5,5,5,5,5,1,0,2.446000e+11,3.154000e+11
2,HBL,Habib Bank Ltd,Banking,166.02,174.37,174.05,206.19,37.87,-1.3991,-0.0657,...,5,5,4,4,5,4,1,0,7.038500e+12,6.815000e+11
3,SNGP,Sui Northern Gas Pipelines,Energy,48.55,47.19,49.94,77.97,53.02,0.4367,-0.0190,...,5,5,4,4,5,4,0,0,6.400000e+11,8.000000e+10
4,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11


Momentum Leaders: 7 rows


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,PSO,Pakistan State Oil,Energy,412.67,395.61,384.48,373.85,62.11,2.4393,0.0981,...,5,5,4,4,5,4,1,1,7.354000e+11,2.500000e+11
1,FFBL,Fauji Fertilizer Bin Qasim,Fertilizer,88.94,82.01,71.11,45.65,62.17,-0.1680,0.1785,...,4,4,3,3,4,3,1,1,NaN,NaN
2,ATLH,Atlas Honda Ltd,Automobile,1398.07,1385.04,1315.31,1086.56,64.11,-4.4715,0.0354,...,3,3,1,1,3,1,1,1,NaN,NaN
3,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11
4,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,5,5,4,4,5,4,1,1,2.213000e+11,7.058000e+11


Low Risk Blue Chips: 18 rows


...

,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,GLAXO,GlaxoSmithKline Pakistan,Pharma,398.73,430.29,423.72,404.55,31.68,-3.4749,-0.1135,...,3,3,0,0,3,0,1,1,NaN,NaN


Avoid / High Risk: 2 rows


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,SSGC,Sui Southern Gas Company,Energy,61.49,66.40,68.02,59.38,25.23,-0.5820,-0.0997,...,5,5,4,4,5,4,0,1,6.100000e+11,-3.000000e+10
1,KTML,Kohinoor Textile Mills,Textiles,64.20,67.19,142.20,165.03,28.09,5.5078,-0.1129,...,4,4,0,0,4,0,0,0,NaN,NaN


Sector Relative Value: 7 rows


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,ABL,Allied Bank Ltd,Banking,165.04,159.21,154.50,161.53,64.50,0.5718,0.1045,...,5,5,4,4,5,4,1,0,1.949900e+12,1.701000e+11
1,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,5,5,4,4,5,4,1,1,2.213000e+11,7.058000e+11
2,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11
3,MLCF,Maple Leaf Cement Factory,Cement,97.50,104.02,103.21,75.39,37.34,-0.9555,-0.0929,...,5,5,5,5,5,5,1,1,5.460000e+10,6.540000e+10
4,NESTLE,Nestle Pakistan Ltd,FMCG,8113.57,8274.93,8336.31,7587.63,40.58,-19.7817,-0.0381,...,2,2,2,2,2,2,0,1,7.000000e+10,4.000000e+10


In [5]:
try:
    apply_preset(df, "Fake Preset Name")
except ValueError as e:
    print("Correctly failed:", e)

Correctly failed: Unknown preset 'Fake Preset Name'. Available presets: Strong Buy Candidates, Undervalued Quality, Dividend Picks, Momentum Leaders, Low Risk Blue Chips, Oversold Bounce, Avoid / High Risk, Sector Relative Value


In [1]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run()

df = engine.get_screener_df()
df.head()

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,5,5,4,4,5,4,1,1,2.213000e+11,7.058000e+11
1,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11
2,OGDC,Oil & Gas Dev. Company,Energy,417.47,402.31,368.23,304.85,67.58,-0.7895,0.0937,...,5,5,4,4,5,4,1,1,2.450000e+11,1.005000e+12
3,ABL,Allied Bank Ltd,Banking,165.04,159.21,154.50,161.53,64.50,0.5718,0.1045,...,5,5,4,4,5,4,1,0,1.949900e+12,1.701000e+11
4,PSO,Pakistan State Oil,Energy,412.67,395.61,384.48,373.85,62.11,2.4393,0.0981,...,5,5,4,4,5,4,1,1,7.354000e+11,2.500000e+11


In [5]:
from src.reports.stock_report import (
    build_stock_report,
    build_analyst_summary,
    report_to_dataframe,
    get_report_warning_messages,
)
from src.screener.engine import ScreenerEngine

In [6]:
engine = ScreenerEngine()
engine.run()
df = engine.get_screener_df()
print(f"Loaded {len(df)} tickers")
df.head()

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals
Loaded 38 tickers


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,5,5,4,4,5,4,1,1,2.213000e+11,7.058000e+11
1,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11
2,OGDC,Oil & Gas Dev. Company,Energy,417.47,402.31,368.23,304.85,67.58,-0.7895,0.0937,...,5,5,4,4,5,4,1,1,2.450000e+11,1.005000e+12
3,ABL,Allied Bank Ltd,Banking,165.04,159.21,154.50,161.53,64.50,0.5718,0.1045,...,5,5,4,4,5,4,1,0,1.949900e+12,1.701000e+11
4,PSO,Pakistan State Oil,Energy,412.67,395.61,384.48,373.85,62.11,2.4393,0.0981,...,5,5,4,4,5,4,1,1,7.354000e+11,2.500000e+11


In [7]:
ticker = df.iloc[0]["ticker"]
report = build_stock_report(df, ticker)
print(report.keys())

dict_keys(['identity', 'scores', 'fundamentals', 'sector_comparison', 'technicals', 'verdict', 'summary', 'warnings'])


In [8]:
print(report["summary"])

Pakistan Petroleum Ltd (PPL) operates in the Energy. Valuation metrics suggest a P/E of 5.49 and a P/B of 0.72. The stock appears cheaper than its sector peers on a P/E basis. Its price-to-book ratio also appears below the sector average. ROE stands at 1,303.5%. Net profit margin is 3,793.8%. Profitability appears weaker than the sector average. The balance sheet appears conservatively leveraged. The overall risk profile appears relatively low. Risk is lower than the sector average. The stock offers a dividend yield of approximately 538.7%. This yield appears above the sector average. The payout ratio appears sustainable. Technical indicators suggest a strong trend. Price momentum has been positive over the past 1 and 3 months. The composite score is meaningfully above the sector average. Sector benchmarking classifies this stock as: 'Attractive vs Sector'. Overall assessment: Buy. Positive factor tilt across technical and fundamental pillars with manageable risk. This summary is based

In [10]:
for w in report.get("warnings", []):
    print(f"⚠ {w}")

In [11]:
report_to_dataframe(report)

,section,metric,value
0,identity,ticker,PPL
1,identity,name,Pakistan Petroleum Ltd
2,identity,sector,Energy
3,identity,latest_close,185.63
4,scores,technical_score,73.53
5,scores,fundamental_score,65.76
6,scores,risk_score,9.77
7,scores,composite_score,73.76
8,fundamentals,pe_ratio,5.49
9,fundamentals,pb_ratio,0.72


In [12]:
report["scores"]

{'technical_score': 73.53,
 'fundamental_score': 65.76,
 'risk_score': 9.77,
 'composite_score': 73.76}

In [13]:
try:
    build_stock_report(df, "FAKE_TICKER")
except ValueError as e:
    print("Correctly failed:", e)

Correctly failed: Ticker 'FAKE_TICKER' not found in screener. Available tickers: ABL, ATLH, BAFL, CHCC, COLG, DGKC, EFERT, ENGRO, FATIMA, FEROZ, FFBL, FFC, GATM, GLAXO, HBL, ICI, INDU, KTML, LUCK, MCB...


In [2]:
import pandas as pd
from src.screener.engine import ScreenerEngine
from src.portfolio.portfolio_builder import (
    select_top_stocks,
    build_equal_weight_portfolio,
    build_inverse_volatility_portfolio,
    build_portfolio,
    summarize_portfolio,
)

engine = ScreenerEngine()
engine.run(force_reseed=False)
df = engine.get_screener_df()

# ── 1. Select top 10 ─────────────────────────────────────────────
selected = select_top_stocks(df, top_n=10, min_composite=50, max_risk=70)
print(selected[["ticker", "composite_score", "risk_score"]])

# ── 2. Equal weight ───────────────────────────────────────────────
eq = build_equal_weight_portfolio(selected)
print(eq[["ticker", "weight", "weighting_method"]])
print("Weight sum:", eq["weight"].sum())

# ── 3. Inverse volatility ─────────────────────────────────────────
iv = build_inverse_volatility_portfolio(selected)
print(iv[["ticker", "weight", "weighting_method"]])
print("Weight sum:", iv["weight"].sum())

# ── 4. One-call build ─────────────────────────────────────────────
port = build_portfolio(df, top_n=8, weighting="inverse_volatility",
                       min_composite=55, max_risk=65)
print(port[["ticker", "sector", "weight"]])

# ── 5. Summary ────────────────────────────────────────────────────
summary = summarize_portfolio(port)
for k, v in summary.items():
    print(f"  {k:<26} {v}")

# ── 6. Edge cases ─────────────────────────────────────────────────
empty_result = build_portfolio(pd.DataFrame(), top_n=5)
print("Empty input handled:", empty_result.empty)

no_match = build_portfolio(df, min_composite=999)
print("No-match handled:", no_match.empty)

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals
   ticker  composite_score  risk_score
0     PPL            73.76        9.77
1     FFC            72.65       18.97
2    OGDC            71.06        7.84
3     ABL            70.81       24.34
4     PSO            65.39       25.57
5  FATIMA            63.80       16.19
6    ATLH            63.34       32.74
7    FFBL            61.52       44.00
8    MLCF            61.09       12.64
9  

In [3]:
from src.portfolio.portfolio_builder import (
    select_top_stocks,
    build_equal_weight_portfolio,
    build_inverse_volatility_portfolio,
    build_portfolio,
    summarize_portfolio,
)

selected = select_top_stocks(df, top_n=10)
display(selected[["ticker", "sector", "composite_score", "risk_score"]])

,ticker,sector,composite_score,risk_score
0,PPL,Energy,73.76,9.77
1,FFC,Fertilizer,72.65,18.97
2,OGDC,Energy,71.06,7.84
3,ABL,Banking,70.81,24.34
4,PSO,Energy,65.39,25.57
5,FATIMA,Fertilizer,63.80,16.19
6,ATLH,Automobile,63.34,32.74
7,FFBL,Fertilizer,61.52,44.00
8,MLCF,Cement,61.09,12.64
9,UBL,Banking,61.06,27.95


In [4]:
from src.portfolio.portfolio_builder import (
    select_top_stocks,
    build_equal_weight_portfolio,
    build_inverse_volatility_portfolio,
    build_portfolio,
    summarize_portfolio,
)

selected = select_top_stocks(df, top_n=10)
display(selected[["ticker", "sector", "composite_score", "risk_score"]])

,ticker,sector,composite_score,risk_score
0,PPL,Energy,73.76,9.77
1,FFC,Fertilizer,72.65,18.97
2,OGDC,Energy,71.06,7.84
3,ABL,Banking,70.81,24.34
4,PSO,Energy,65.39,25.57
5,FATIMA,Fertilizer,63.80,16.19
6,ATLH,Automobile,63.34,32.74
7,FFBL,Fertilizer,61.52,44.00
8,MLCF,Cement,61.09,12.64
9,UBL,Banking,61.06,27.95


In [6]:
from src.portfolio.portfolio_builder import (
    select_top_stocks,
    build_equal_weight_portfolio,
    build_inverse_volatility_portfolio,
    build_portfolio,
    summarize_portfolio,
)

selected = select_top_stocks(df, top_n=10)
display(selected[["ticker", "sector", "composite_score", "risk_score"]])

,ticker,sector,composite_score,risk_score
0,PPL,Energy,73.76,9.77
1,FFC,Fertilizer,72.65,18.97
2,OGDC,Energy,71.06,7.84
3,ABL,Banking,70.81,24.34
4,PSO,Energy,65.39,25.57
5,FATIMA,Fertilizer,63.80,16.19
6,ATLH,Automobile,63.34,32.74
7,FFBL,Fertilizer,61.52,44.00
8,MLCF,Cement,61.09,12.64
9,UBL,Banking,61.06,27.95


In [7]:
p_equal = build_equal_weight_portfolio(selected)
display(p_equal[["ticker", "weight", "weighting_method", "composite_score"]])
print(p_equal["weight"].sum())

,ticker,weight,weighting_method,composite_score
0,PPL,0.1,equal,73.76
1,FFC,0.1,equal,72.65
2,OGDC,0.1,equal,71.06
3,ABL,0.1,equal,70.81
4,PSO,0.1,equal,65.39
5,FATIMA,0.1,equal,63.80
6,ATLH,0.1,equal,63.34
7,FFBL,0.1,equal,61.52
8,MLCF,0.1,equal,61.09
9,UBL,0.1,equal,61.06


1.0


In [8]:
p_inv = build_inverse_volatility_portfolio(selected)
display(p_inv[[c for c in ["ticker", "weight", "weighting_method", "volatility_30d", "volatility"] if c in p_inv.columns]])
print(p_inv["weight"].sum())

,ticker,weight,weighting_method,volatility_30d
0,PPL,0.085453,inverse_volatility,0.3316
1,FFC,0.131246,inverse_volatility,0.2159
2,OGDC,0.101928,inverse_volatility,0.2780
3,ABL,0.135514,inverse_volatility,0.2091
4,PSO,0.105574,inverse_volatility,0.2684
5,FATIMA,0.070911,inverse_volatility,0.3996
6,ATLH,0.151287,inverse_volatility,0.1873
7,FFBL,0.043474,inverse_volatility,0.6518
8,MLCF,0.091613,inverse_volatility,0.3093
9,UBL,0.083000,inverse_volatility,0.3414


1.0


In [9]:
summary = summarize_portfolio(p_equal)
summary

{'holdings': 10,
 'weight_sum': 1.0,
 'weighting_method': 'equal',
 'avg_composite_score': 66.45,
 'avg_technical_score': 68.46,
 'avg_fundamental_score': 58.66,
 'avg_risk_score': 22.0,
 'avg_dividend_yield': 3.86,
 'top_holding': 'PPL',
 'sector_allocation': {'Energy': 0.3,
  'Fertilizer': 0.3,
  'Banking': 0.2,
  'Automobile': 0.1,
  'Cement': 0.1}}

In [1]:
import pandas as pd
from src.screener.engine import ScreenerEngine
from src.portfolio.portfolio_builder import build_portfolio
from src.portfolio.backtest import (
    build_price_matrix,
    calculate_portfolio_returns,
    calculate_max_drawdown,
    backtest_portfolio,
    backtest_to_dataframe,
)

engine = ScreenerEngine()
engine.run(force_reseed=False)
screener_df = engine.get_screener_df()

# ── Build a portfolio ─────────────────────────────────────────────
port = build_portfolio(screener_df, top_n=8, weighting="inverse_volatility")
print(port[["ticker", "weight"]])

# ── 1. Price matrix ───────────────────────────────────────────────
tickers = port["ticker"].tolist()
matrix  = build_price_matrix(engine, tickers)
print(matrix.tail())

# ── 2. Daily returns ──────────────────────────────────────────────
weights = dict(zip(port["ticker"], port["weight"]))
daily   = calculate_portfolio_returns(matrix, weights)
print(daily.tail())

# ── 3. Max drawdown ───────────────────────────────────────────────
cum = (1 + daily).cumprod()
mdd = calculate_max_drawdown(cum)
print(f"Max drawdown: {mdd*100:.2f}%")

# ── 4. Full backtest ─────────────────────────────────────────────
result = backtest_portfolio(engine, port)
for k, v in result.items():
    if not isinstance(v, pd.Series):
        print(f"  {k:<26} {v}")

# ── 5. Result as DataFrame ────────────────────────────────────────
bt_df = backtest_to_dataframe(result)
display(bt_df)

# ── Edge cases ────────────────────────────────────────────────────
r_empty = backtest_portfolio(engine, pd.DataFrame())
print("Empty portfolio handled:", r_empty["status"])

r_bad = backtest_portfolio(engine, pd.DataFrame({"ticker": ["ZZZZ"], "weight": [1.0]}))
print("Unknown ticker handled:", r_bad["status"])

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals
   ticker    weight
0     PPL  0.103530
1     FFC  0.159012
2    OGDC  0.123491
3     ABL  0.164183
4     PSO  0.127908
5  FATIMA  0.085912
6    ATLH  0.183292
7    FFBL  0.052671
[backtest] Price matrix: 8 ticker(s) | 2324 rows | 2017-01-02 → 2026-04-24
               PPL  FFC    OGDC     ABL     PSO  FATIMA  ATLH  FFBL
date                                                               
20

C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:192: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change()
C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:192: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change()


,metric,value
0,Status,ok
1,Start Date,2017-01-03
2,End Date,2026-04-24
3,Trading Days,2323
4,Cumulative Return,+227.54%
5,Annualized Return,+13.74%
6,Annualized Volatility,+23.00%
7,Sharpe Ratio,0.5973
8,Max Drawdown,-48.07%
9,Tickers,"PPL, FFC, OGDC, ABL, PSO, FATIMA, ATLH, FFBL"


Empty portfolio handled: error
Unknown ticker handled: error


C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:292: UserWarning: [backtest] portfolio_df is empty — cannot backtest.
  warnings.warn(f"[backtest] {msg}")
C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:83: UserWarning: [backtest] No price data for ZZZZ — skipping.
  warnings.warn(
C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:110: UserWarning: [backtest] build_price_matrix: no price data loaded.
  warnings.warn("[backtest] build_price_matrix: no price data loaded.")
C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:292: UserWarning: [backtest] Could not build price matrix — no price data available.
  warnings.warn(f"[backtest] {msg}")


In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
from src.screener.engine import ScreenerEngine
from src.portfolio.portfolio_builder import build_portfolio
from src.portfolio.backtest import (
    build_price_matrix,
    calculate_portfolio_returns,
    calculate_max_drawdown,
    backtest_portfolio,
    backtest_to_dataframe,
)

engine = ScreenerEngine()
engine.run(force_reseed=False)
screener_df = engine.get_screener_df()

# ── Build a portfolio ─────────────────────────────────────────────
port = build_portfolio(screener_df, top_n=8, weighting="inverse_volatility")
print(port[["ticker", "weight"]])

# ── 1. Price matrix ───────────────────────────────────────────────
tickers = port["ticker"].tolist()
matrix  = build_price_matrix(engine, tickers)
print(matrix.tail())

# ── 2. Daily returns ──────────────────────────────────────────────
weights = dict(zip(port["ticker"], port["weight"]))
daily   = calculate_portfolio_returns(matrix, weights)
print(daily.tail())

# ── 3. Max drawdown ───────────────────────────────────────────────
cum = (1 + daily).cumprod()
mdd = calculate_max_drawdown(cum)
print(f"Max drawdown: {mdd*100:.2f}%")

# ── 4. Full backtest ─────────────────────────────────────────────
result = backtest_portfolio(engine, port)
for k, v in result.items():
    if not isinstance(v, pd.Series):
        print(f"  {k:<26} {v}")

# ── 5. Result as DataFrame ────────────────────────────────────────
bt_df = backtest_to_dataframe(result)
display(bt_df)

# ── Edge cases ────────────────────────────────────────────────────
r_empty = backtest_portfolio(engine, pd.DataFrame())
print("Empty portfolio handled:", r_empty["status"])

r_bad = backtest_portfolio(engine, pd.DataFrame({"ticker": ["ZZZZ"], "weight": [1.0]}))
print("Unknown ticker handled:", r_bad["status"])

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals
   ticker    weight
0     PPL  0.103530
1     FFC  0.159012
2    OGDC  0.123491
3     ABL  0.164183
4     PSO  0.127908
5  FATIMA  0.085912
6    ATLH  0.183292
7    FFBL  0.052671
[backtest] Price matrix: 8 ticker(s) | 2324 rows | 2017-01-02 → 2026-04-24
               PPL  FFC    OGDC     ABL     PSO  FATIMA  ATLH  FFBL
date                                                               
20

C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:192: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change()
C:\Users\berdy\Desktop\PSX-40-STOCK-SCREENER\src\portfolio\backtest.py:192: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change()


,metric,value
0,Status,ok
1,Start Date,2017-01-03
2,End Date,2026-04-24
3,Trading Days,2323
4,Cumulative Return,+227.54%
5,Annualized Return,+13.74%
6,Annualized Volatility,+23.00%
7,Sharpe Ratio,0.5973
8,Max Drawdown,-48.07%
9,Tickers,"PPL, FFC, OGDC, ABL, PSO, FATIMA, ATLH, FFBL"


Empty portfolio handled: error
Unknown ticker handled: error


In [1]:
from src.screener.engine import ScreenerEngine

engine = ScreenerEngine()
engine.run()

df = engine.get_screener_df()
df.head()

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_composite_score_count,sector_valid_dividend_yield_count,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity
0,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,5,5,4,4,5,4,1,1,2.213000e+11,7.058000e+11
1,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,4,4,3,3,4,3,1,1,2.925000e+11,1.350000e+11
2,OGDC,Oil & Gas Dev. Company,Energy,417.47,402.31,368.23,304.85,67.58,-0.7895,0.0937,...,5,5,4,4,5,4,1,1,2.450000e+11,1.005000e+12
3,ABL,Allied Bank Ltd,Banking,165.04,159.21,154.50,161.53,64.50,0.5718,0.1045,...,5,5,4,4,5,4,1,0,1.949900e+12,1.701000e+11
4,PSO,Pakistan State Oil,Energy,412.67,395.61,384.48,373.85,62.11,2.4393,0.0981,...,5,5,4,4,5,4,1,1,7.354000e+11,2.500000e+11


In [2]:
from src.portfolio.portfolio_builder import (
    select_top_stocks,
    build_equal_weight_portfolio,
    build_inverse_volatility_portfolio,
    build_portfolio,
    summarize_portfolio,
)

selected = select_top_stocks(df, top_n=10)

p_equal = build_equal_weight_portfolio(selected)

display(p_equal[[
    c for c in [
        "ticker",
        "name",
        "sector",
        "weight",
        "weighting_method",
        "composite_score",
        "risk_score"
    ]
    if c in p_equal.columns
]])

print("Weight sum:", p_equal["weight"].sum())

,ticker,name,sector,weight,weighting_method,composite_score,risk_score
0,PPL,Pakistan Petroleum Ltd,Energy,0.1,equal,73.76,9.77
1,FFC,Fauji Fertilizer Company,Fertilizer,0.1,equal,72.65,18.97
2,OGDC,Oil & Gas Dev. Company,Energy,0.1,equal,71.06,7.84
3,ABL,Allied Bank Ltd,Banking,0.1,equal,70.81,24.34
4,PSO,Pakistan State Oil,Energy,0.1,equal,65.39,25.57
5,FATIMA,Fatima Fertilizer Company,Fertilizer,0.1,equal,63.80,16.19
6,ATLH,Atlas Honda Ltd,Automobile,0.1,equal,63.34,32.74
7,FFBL,Fauji Fertilizer Bin Qasim,Fertilizer,0.1,equal,61.52,44.00
8,MLCF,Maple Leaf Cement Factory,Cement,0.1,equal,61.09,12.64
9,UBL,United Bank Ltd,Banking,0.1,equal,61.06,27.95


Weight sum: 1.0


In [3]:
from src.portfolio.backtest import (
    build_price_matrix,
    calculate_portfolio_returns,
    calculate_max_drawdown,
    backtest_portfolio,
    backtest_to_dataframe,
)

tickers = p_equal["ticker"].tolist()

price_matrix = build_price_matrix(engine, tickers)

display(price_matrix.head())
display(price_matrix.tail())
print(price_matrix.shape)

[backtest] Price matrix: 10 ticker(s) | 2324 rows | 2017-01-02 → 2026-04-24


,PPL,FFC,OGDC,ABL,PSO,FATIMA,ATLH,FFBL,MLCF,UBL
date,,,,,,,,,,
2017-01-02,186.99,107.54,166.15,120.47,444.52,37.50,590.00,52.10,126.15,243.59
2017-01-03,192.06,110.79,170.74,121.00,461.59,37.96,587.28,53.52,131.38,249.48
2017-01-04,192.52,109.46,168.96,121.46,460.69,37.59,588.84,52.98,131.88,247.08
2017-01-05,193.27,111.35,169.27,119.70,456.40,37.13,607.78,53.60,129.67,246.82
2017-01-06,192.85,114.02,170.19,120.99,460.65,38.96,608.29,53.81,129.04,247.75


,PPL,FFC,OGDC,ABL,PSO,FATIMA,ATLH,FFBL,MLCF,UBL
date,,,,,,,,,,
2026-04-20,185.22,NaN,408.45,159.78,408.55,NaN,NaN,NaN,NaN,364.32
2026-04-21,184.24,NaN,416.49,161.56,423.28,NaN,NaN,NaN,NaN,359.18
2026-04-22,188.99,NaN,418.20,165.95,416.47,NaN,NaN,NaN,NaN,366.45
2026-04-23,189.63,NaN,418.63,168.02,415.95,NaN,NaN,NaN,NaN,365.88
2026-04-24,185.63,NaN,417.47,165.04,412.67,NaN,NaN,NaN,NaN,371.11


(2324, 10)


In [1]:
from src.screener.engine import ScreenerEngine
from src.portfolio.portfolio_builder import select_top_stocks, build_equal_weight_portfolio
from src.portfolio.backtest import build_price_matrix, backtest_portfolio, backtest_to_dataframe

engine = ScreenerEngine()
engine.run()

df = engine.get_screener_df()

selected = select_top_stocks(df, top_n=10)
p_equal = build_equal_weight_portfolio(selected)

price_matrix = build_price_matrix(engine, p_equal["ticker"].tolist())
result = backtest_portfolio(engine, p_equal)

display(p_equal)
display(price_matrix.head())
display(backtest_to_dataframe(result))

print(result["metrics"])
print(result["warnings"])

[engine] Initialising database...
[engine] DB has data for 38 ticker(s) — skipping seed.
[engine] Loading universe...
[engine] Computing technical indicators...
[engine] Merging fundamental metrics...
[fundamental] Built metrics for 53 ticker(s) | avg quality score: 81.3/100
[engine] Fundamental merge — 23/38 tickers have ratio data.
[engine] Running three-pillar scoring...
[engine] Applying sector benchmarks...
[sector_benchmark] Applied benchmarks — 38 stocks across 10 sectors | 38 stocks labelled.
[engine] Applying verdicts...
[engine] Ready — 38 tickers | 80,701 price rows | 40 with fundamentals
[backtest] Price matrix: 10 ticker(s) | 2324 rows | 2017-01-02 → 2026-04-24
[backtest] Price matrix: 10 ticker(s) | 2324 rows | 2017-01-02 → 2026-04-24
[backtest] Done — 2323 days | cumulative: +232.60% | max drawdown: -57.87%


,ticker,name,sector,latest_close,sma_20,sma_50,sma_200,rsi_14,macd_hist,return_1m,...,sector_valid_pb_count,sector_valid_pe_count,sector_valid_risk_score_count,sector_valid_roe_count,sma20_above_sma50,sma50_above_sma200,total_debt,total_equity,weight,weighting_method
0,PPL,Pakistan Petroleum Ltd,Energy,185.63,180.76,177.04,174.44,57.39,0.8882,0.0692,...,4,4,5,4,1,1,2.213000e+11,7.058000e+11,0.1,equal
1,FFC,Fauji Fertilizer Company,Fertilizer,477.28,476.48,457.92,408.22,55.45,-0.9786,0.0794,...,3,3,4,3,1,1,2.925000e+11,1.350000e+11,0.1,equal
2,OGDC,Oil & Gas Dev. Company,Energy,417.47,402.31,368.23,304.85,67.58,-0.7895,0.0937,...,4,4,5,4,1,1,2.450000e+11,1.005000e+12,0.1,equal
3,ABL,Allied Bank Ltd,Banking,165.04,159.21,154.50,161.53,64.50,0.5718,0.1045,...,4,4,5,4,1,0,1.949900e+12,1.701000e+11,0.1,equal
4,PSO,Pakistan State Oil,Energy,412.67,395.61,384.48,373.85,62.11,2.4393,0.0981,...,4,4,5,4,1,1,7.354000e+11,2.500000e+11,0.1,equal
5,FATIMA,Fatima Fertilizer Company,Fertilizer,137.99,140.24,131.09,100.61,51.92,-0.8867,0.0952,...,3,3,4,3,1,1,9.540000e+10,1.200000e+11,0.1,equal
6,ATLH,Atlas Honda Ltd,Automobile,1398.07,1385.04,1315.31,1086.56,64.11,-4.4715,0.0354,...,1,1,3,1,1,1,NaN,NaN,0.1,equal
7,FFBL,Fauji Fertilizer Bin Qasim,Fertilizer,88.94,82.01,71.11,45.65,62.17,-0.1680,0.1785,...,3,3,4,3,1,1,NaN,NaN,0.1,equal
8,MLCF,Maple Leaf Cement Factory,Cement,97.50,104.02,103.21,75.39,37.34,-0.9555,-0.0929,...,5,5,5,5,1,1,5.460000e+10,6.540000e+10,0.1,equal
9,UBL,United Bank Ltd,Banking,371.11,363.39,333.61,342.88,61.39,-1.1399,0.1205,...,4,4,5,4,1,0,4.824900e+12,3.851000e+11,0.1,equal


,PPL,FFC,OGDC,ABL,PSO,FATIMA,ATLH,FFBL,MLCF,UBL
date,,,,,,,,,,
2017-01-02,186.99,107.54,166.15,120.47,444.52,37.50,590.00,52.10,126.15,243.59
2017-01-03,192.06,110.79,170.74,121.00,461.59,37.96,587.28,53.52,131.38,249.48
2017-01-04,192.52,109.46,168.96,121.46,460.69,37.59,588.84,52.98,131.88,247.08
2017-01-05,193.27,111.35,169.27,119.70,456.40,37.13,607.78,53.60,129.67,246.82
2017-01-06,192.85,114.02,170.19,120.99,460.65,38.96,608.29,53.81,129.04,247.75


,metric,value
0,Status,ok
1,Cumulative Return,+232.60%
2,Annualized Return,+13.92%
3,Annualized Volatility,+25.90%
4,Sharpe Ratio,0.5375
5,Max Drawdown,-57.87%
6,Start Date,2017-01-03
7,End Date,2026-04-24
8,Trading Days,2323
9,Tickers,"PPL, FFC, OGDC, ABL, PSO, FATIMA, ATLH, FFBL, ..."


{'cumulative_return': 2.326, 'annualized_return': 0.1392, 'annualized_volatility': 0.259, 'sharpe_ratio': 0.5375, 'max_drawdown': -0.5787, 'start_date': '2017-01-03', 'end_date': '2026-04-24', 'trading_days': 2323, 'tickers': ['PPL', 'FFC', 'OGDC', 'ABL', 'PSO', 'FATIMA', 'ATLH', 'FFBL', 'MLCF', 'UBL']}
[]
